# Writing a quantum program 2 — component properties

Compose reusable components while keeping signatures, compiler recipes, simulation and drawing with their existing FlagQuantum owners. This tutorial follows all cells of the frozen component tutorial.

## Negating an integer again

The signed MSB-first transformation is complement followed by AddK(1). The caller owns all target, clean work and measurement slots.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from importlib import import_module
from typing import cast

import numpy as np
from numpy.typing import NDArray

import flagquantum as fq
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.qec_gate_cost import (
    QECGateCost,
    query_qec_gate_cost,
)
from flagquantum.compiler._component_trace.resource_width import (
    query_resource_logical_width,
)
from flagquantum.compiler._component_trace.symbolic_math import SymbolicInt, smax
from flagquantum.compiler.registers.arithmetic.addition import negate_ir
from flagquantum.compiler.registers.arithmetic.arithmetic_declarations import (
    arithmetic_call_graph,
)
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.core.ir.program.register_types import signed_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_graph import export_register_graph_dot
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.simulation.classical_negation import negate_classical_value
from flagquantum.simulation.tensor_network.classical_action_tensor import (
    classical_action_tensor,
)

width = 8
target = tuple(range(width))
constant = tuple(range(width, 2 * width))
carries = tuple(range(2 * width, 3 * width - 1))
classical_bits = tuple(range(width - 1))
initial_instructions = negate_ir(target, constant, carries, classical_bits)
initial_ir = CircuitIR(3 * width - 1, initial_instructions)
initial_ir.validate()


## Declare a component

A frozen Python value holds compile-time parameters. Core Signature declares the logical through-register. A declaration can be inspected before an implementation is attached.

In [ ]:
@dataclass(frozen=True)
class NegationDeclaration:
    """Declare the logical signed through-register before decomposition."""

    n: SymbolicInt = 8

    @property
    def signature(self) -> Signature:
        return Signature((Register("x", signed_register_type(self.n)),))


Instantiate the declaration to query its logical interface.

In [ ]:
declaration = NegationDeclaration()


Inspect the number of named registers.

In [ ]:
print(len(declaration.signature))


The signature-only qubit estimate is eight. It omits implementation workspace and can underestimate the complete peak.

In [ ]:
signature_only_width = sum(r.total_bits() for r in declaration.signature)
print(signature_only_width)


## Implement the component

The notebook component composes existing compiler recipes through the append protocol. Its two direct calls remain available separately for hierarchical analysis. Concrete emission binds n target wires, n zero constant-load wires, n−1 zero carry wires and n−1 caller-owned classical slots. All work is restored to zero. The circuit has 3n−1 wires. Simulation and resource evaluation use their existing owners.

In [ ]:
@dataclass(frozen=True)
class Negation(NegationDeclaration):
    """Bind existing domain operations without owning numerical algorithms."""

    @property
    def calls(self) -> tuple[ArithmeticCall, ArithmeticCall]:
        return (
            ArithmeticCall("bitwise_not", (self.n,), ("twos_complement",)),
            ArithmeticCall("constant_add", (self.n,), ("twos_complement",), constant=1),
        )

    def instructions(self) -> tuple[Instruction, ...]:
        if type(self.n) is not int or self.n < 1:
            raise ValueError("Physical emission requires a positive integer width")
        return negate_ir(
            tuple(range(self.n)),
            tuple(range(self.n, 2 * self.n)),
            tuple(range(2 * self.n, 3 * self.n - 1)),
            tuple(range(self.n - 1)),
        )

    def append_to(self, circuit: fq.Circuit) -> fq.Circuit:
        instructions = self.instructions()
        return circuit.compose(CircuitIR(3 * int(self.n) - 1, instructions))

    def gate_cost(self) -> QECGateCost:
        return sum(
            (
                query_qec_gate_cost(call, declared_calls=arithmetic_call_graph(call))
                for call in self.calls
            ),
            QECGateCost(),
        )

    def logical_width(self) -> SymbolicInt:
        return smax(
            self.n,
            *(
                query_resource_logical_width(
                    call, declared_calls=arithmetic_call_graph(call)
                )
                for call in self.calls
            )
        )

    def classical(self, x: int) -> int:
        if type(self.n) is not int:
            raise ValueError("Classical execution requires a concrete integer width")
        return negate_classical_value(x, self.n, signed=True)

    def tensor(self) -> NDArray[np.float64]:
        def action(**values: object) -> tuple[int]:
            return (self.classical(cast(int, values["x"])),)

        return classical_action_tensor(
            self.signature, action, operation_name="Negation"
        )


## Inspect the component

Instantiate the implemented component.

In [ ]:
negate = Negation()
circuit = fq.Circuit(23).append(negate)
assert circuit.to_ir().instructions == initial_instructions


Draw the opaque component with its declared logical width.

In [ ]:
ipython_display = import_module("IPython.display")
opaque_dot = export_call_graph_dot(
    "Negation", {}, details=lambda node: f"x signed[{negate.n}]"
)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(opaque_dot)))


A component stays hierarchical until its direct calls are requested. The graph below exposes precisely the complement and AddK(1) dependency.

In [ ]:
decomposition_score = layout_register_score(
    {"x": 8},
    ("x",),
    (
        RegisterStep("BitwiseNot", ("x",), ("x",)),
        RegisterStep("AddK(1)", ("x",), ("x",)),
    ),
    outputs=("x",),
)
decomposition_dot = export_register_graph_dot(decomposition_score)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(decomposition_dot)))


Concrete `instructions()` emits the complete one- and two-qubit primitive stream, including measurement, feedback and reset. Hierarchical queries avoid constructing that stream for large or symbolic widths.

## Analyze the component

Query the existing recursive QEC gate models independently of actual emitted primitives.

In [ ]:
gate_model = negate.gate_cost()
print(gate_model)
primitive_counts = Counter(g.name for g in circuit.to_ir().instructions)
print(dict(primitive_counts))


In [ ]:
peak_width = negate.logical_width()
print(peak_width)


Draw the direct call graph with each component’s independently queried cost. The direct dependency graph and resource graph serve different purposes.

In [ ]:
def call_title(node: str | ArithmeticCall) -> str:
    if isinstance(node, str):
        return node
    label = {"bitwise_not": "BitwiseNot", "constant_add": "AddK(1)"}[node.operation]
    return f"{label} signed[{node.widths[0]}]"


call_edges: dict[tuple[str | ArithmeticCall, str | ArithmeticCall], SymbolicInt] = {
    ("Negation", call): 1 for call in negate.calls
}
call_dot = export_call_graph_dot(
    "Negation",
    call_edges,
    title=call_title,
    details=lambda node: (
        str(query_qec_gate_cost(node, declared_calls=arithmetic_call_graph(node)))
        if isinstance(node, ArithmeticCall)
        else str(negate.gate_cost())
    ),
)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(call_dot)))


Evaluate the shared reversible classical action.

In [ ]:
negated_value = negate.classical(5)
print(negated_value)


Construct the phase-free dense tensor using the shared signed register codecs and action enumerator. This eight-bit tensor has 256×256 entries. Quantum branch evidence separately verifies that the emitted implementation has the same phases.

In [ ]:
unitary = negate.tensor()
print(unitary.shape)


## A family of components

The compile-time width n controls the signed signature and both children. The immutable class already accepts this parameter.

In [ ]:
family_example = Negation(n=4)
assert family_example.signature[0].bitsize == 4


Large-width queries preserve the two-child hierarchy and avoid emitting a physical circuit.

In [ ]:
large = Negation(n=2048)
large_dot = export_call_graph_dot(
    "Negation", {}, details=lambda node: f"x signed[{large.n}]"
)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(large_dot)))


In [ ]:
large_edges: dict[tuple[str | ArithmeticCall, str | ArithmeticCall], SymbolicInt] = {
    ("Negation", call): 1 for call in large.calls
}
large_decomposition_dot = export_call_graph_dot(
    "Negation", large_edges, title=call_title
)
ipython_display.display(
    ipython_display.SVG(render_call_graph_svg(large_decomposition_dot))
)


## Symbolic parameters

Core register types and the existing arithmetic resource identities retain symbolic widths. Drawing and resource analysis work without allocating symbolic wires. Physical emission requires a positive concrete integer.

In [ ]:
import sympy

n = sympy.Symbol("n")
symbolic = Negation(n=n)
symbolic_edges: dict[tuple[str | ArithmeticCall, str | ArithmeticCall], SymbolicInt] = {
    ("Negation", call): 1 for call in symbolic.calls
}
symbolic_dot = export_call_graph_dot("Negation", symbolic_edges, title=call_title)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(symbolic_dot)))


The shared QEC cost queries retain symbolic expressions. Their estimates remain separate from decomposed primitive multiplicities.

In [ ]:
symbolic_cost = symbolic.gate_cost()
print(symbolic_cost)


## Provenance and execution scope

Copyright 2024 Google LLC. SPDX-License-Identifier Apache-2.0. Adapted from Qualtran v0.7.0 `tutorials/quantum-programming/2-bloqs.ipynb`, frozen revision `bcb17bb25360ed68ce3bc084645db3e929799389`, Git blob `8ab4e2b48c82b0104d6b89340128d21386a5fa8f`. Every original cell is mapped in order. Native dataclasses and explicit workspace replace the source builder and implicit allocator. The notebook-local class delegates to existing domain owners. The eight-bit tensor enumerates a known phase-free action. Exhaustive emitted quantum histories are validated at three bits. The one-bit signed source constant-load defect remains separately documented. Large and symbolic examples analyze hierarchy without gate emission. Graphviz and IPython are optional presentation dependencies.